In [ ]:
import numpy as np
import pandas as pd
import os

In [ ]:
SEED = 11
B = 5000
Z = {90: 1.645, 95: 1.96, 99: 2.576}
ALPHA = {90: .10, 95: .05, 99: .01}

def bootstrap(data, stat, seed=SEED, B=B):
    rng = np.random.default_rng(seed)
    n = len(data)
    return np.array([stat(data[rng.integers(0, n, n)]) for _ in range(B)])

def normal_cis(est, bs, null):
    se = np.std(bs, ddof=1)
    print(f"Estimate = {est:.6f}, Bootstrap SE = {se:.6f}, (seed={SEED}, B={B})")
    rows = []
    for L, z in Z.items():
        lo, hi = est - z*se, est + z*se
        rows.append({"Level": f"{L}%", "alpha": ALPHA[L], "Lower": lo, "Upper": hi,
                     "H0 value": null,
                     "Decision": "Reject" if (null < lo or null > hi) else "Fail to reject"})
    return pd.DataFrame(rows).round(4)

In [ ]:
ames = pd.read_csv("./data/ames_prices.csv")
ames["central_air"] = (ames["Central.Air"] == "Y").astype(int)
y = ames["central_air"].values
p_hat = y.mean()
print(f"n = {len(y)}, proportion w/ central air = {p_hat:.4f}")

n = 2930, proportion with central air = 0.9331


In [4]:
bs7 = bootstrap(y, np.mean)                 
normal_cis(p_hat, bs7, 0.90)

Estimate = 0.933106, Bootstrap SE = 0.004538, (seed=11, B=5000)


,Level,alpha,Lower,Upper,H0 value,Decision
0,90%,0.10,0.9256,0.9406,0.9,Reject
1,95%,0.05,0.9242,0.9420,0.9,Reject
2,99%,0.01,0.9214,0.9448,0.9,Reject


7. 

93% of homes have central air and all three intervals are >0.90 so we reject H0.

In [ ]:
#8
pairs = np.column_stack([np.arcsinh(ames["area"].values),
                         np.arcsinh(ames["price"].values)])
corr = lambda m: np.corrcoef(m[:, 0], m[:, 1])[0, 1]
r_hat = corr(pairs)
print(f"pearson r = {r_hat:.4f}")

pearson r = 0.7232


In [6]:
bs8 = bootstrap(pairs, corr)
normal_cis(r_hat, bs8, 0)

Estimate = 0.723166, Bootstrap SE = 0.009982, (seed=11, B=5000)


,Level,alpha,Lower,Upper,H0 value,Decision
0,90%,0.10,0.7067,0.7396,0,Reject
1,95%,0.05,0.7036,0.7427,0,Reject
2,99%,0.01,0.6975,0.7489,0,Reject


there arent any zeros in any of the intervals and the correlation is 0.72, and the H0 is rejected at all three levels. it seems like the bigger the home is, the more expensive it is but it also depends on location, the material of the house, environment, etc... The pairs have to stay together because the correlation measures how they move together in a home.

In [7]:
#9

meta = pd.read_csv("./data/metabric.csv")
t = meta["Overall Survival (Months)"].dropna().values
m_hat = t.mean()
print(f"n = {len(t)}, mean = {m_hat:.2f} months = {m_hat/12:.2f} years")

n = 1343, mean = 129.09 months = 10.76 years


In [9]:
bs9 = bootstrap(t, np.mean)
res9 = normal_cis(m_hat, bs9, 120)
res9["Lower(yrs)"] = (res9["Lower"]/12).round(2)
res9["Upper(yrs)"] = (res9["Upper"]/12).round(2)
res9

Estimate = 129.094291, Bootstrap SE = 2.121056, (seed=11, B=5000)


,Level,alpha,Lower,Upper,H0 value,Decision,Lower(yrs),Upper(yrs)
0,90%,0.10,125.6052,132.5834,120,Reject,10.47,11.05
1,95%,0.05,124.9370,133.2516,120,Reject,10.41,11.10
2,99%,0.01,123.6304,134.5581,120,Reject,10.30,11.21


the mean of the recorded time is 129.1 months and the 95% CI is 124.9-133.3 months, and since the intervals are above 120 we reject H0 at every level. this is the average of the time till death and not the actual lifetime mean.

In [11]:
#10

nh = pd.read_csv("./data/nhanes_data_17_18.csv", low_memory=False)
sleep = nh.loc[(nh["AgeInYearsAtScreening"] >= 18) &nh["SleepHoursWeekdaysOrWorkdays"].notna(),
               "SleepHoursWeekdaysOrWorkdays"].astype(float).values
s_hat = sleep.mean()
print(f"n = {len(sleep)}, mean weekday sleep = {s_hat:.3f} hours")

n = 5489, mean weekday sleep = 7.621 hours


In [12]:
bs10 = bootstrap(sleep, np.mean)
normal_cis(s_hat, bs10, 8)

Estimate = 7.621242, Bootstrap SE = 0.022623, (seed=11, B=5000)


,Level,alpha,Lower,Upper,H0 value,Decision
0,90%,0.10,7.5840,7.6585,8,Reject
1,95%,0.05,7.5769,7.6656,8,Reject
2,99%,0.01,7.5630,7.6795,8,Reject


mean of 7.62 hours of weekday sleep and since all intervals are <8 we reject H0. the gap is only 0.38 hours so theres a practical significance.